In [2]:
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import img_to_array, load_img, to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm import tqdm
import tensorflow as tf
from sklearn.metrics import classification_report

# Define paths to dataset folders (update these paths as necessary)
content_paths = ['/Users/samarthuday/Desktop/Git/Art_Style/W0', '/Users/samarthuday/Desktop/Git/Art_Style/W19']
style_path = '/Users/samarthuday/Desktop/Git/Art_Style/Artworks'

# Function to load images from multiple directories and resize them
def load_images_from_folders(folder_paths, target_size=(128, 128), max_images=1000):
    images = []
    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')  # Specify valid image extensions
    for folder_path in folder_paths if isinstance(folder_paths, list) else [folder_paths]:
        for filename in tqdm(os.listdir(folder_path)[:max_images]):
            if filename.lower().endswith(valid_extensions):
                img_path = os.path.join(folder_path, filename)
                try:
                    img = load_img(img_path, target_size=target_size)
                    img_array = img_to_array(img) / 255.0  # Normalize images
                    images.append(img_array)
                except Exception as e:
                    print(f"Could not load image {img_path}: {e}")
            else:
                print(f"Skipping non-image file: {filename}")
    return np.array(images)

# Load content and style images
print("Loading content images...")
content_images = load_images_from_folders(content_paths)
print("Loading style images...")
style_images = load_images_from_folders(style_path)

# Train/test split only for content images
content_train, content_test = train_test_split(content_images, test_size=0.2, random_state=42)

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Model Building
def build_complex_cnn(input_shape, num_classes):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.5),  # Regularization
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Assuming 3 classes for simplicity; adjust based on actual number
num_classes = 3  
input_shape = (128, 128, 3)
model = build_complex_cnn(input_shape, num_classes)

# Dummy one-hot encoded labels
y_train = to_categorical(np.random.randint(0, num_classes, size=(len(content_train),)), num_classes)
y_test = to_categorical(np.random.randint(0, num_classes, size=(len(content_test),)), num_classes)

# Training the Model with data augmentation
epochs = 10
batch_size = 32
history = model.fit(datagen.flow(content_train, y_train, batch_size=batch_size),
                    validation_data=(content_test, y_test),
                    epochs=epochs)

# Model Evaluation
loss, accuracy = model.evaluate(content_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Generate classification report
y_pred = model.predict(content_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

print(classification_report(y_test_classes, y_pred_classes))

# Plot Training History
def plot_training_history(history):
    plt.figure(figsize=(12, 8))

    # Loss Plot
    plt.subplot(2, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss')

    # Accuracy Plot
    plt.subplot(2, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Training and Validation Accuracy')

    # Precision, Recall, and F1 Score
    plt.subplot(2, 2, 3)
    precision = np.sum(y_pred_classes == y_test_classes) / np.sum(y_pred_classes)
    recall = np.sum(y_pred_classes == y_test_classes) / np.sum(y_test_classes)
    f1_score = 2 * (precision * recall) / (precision + recall)
    plt.bar(['Precision', 'Recall', 'F1 Score'], [precision, recall, f1_score])
    plt.title('Precision, Recall, and F1 Score')

    plt.tight_layout()
    plt.show()

plot_training_history(history)

# Visualization of Content, Style, and Stylized Output
def visualize_content_style_output(content_images, style_images, num_samples=3):
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples * 5))
    fig.suptitle('Content Image, Style Image, and Stylized Output', fontsize=16)

    for i in range(num_samples):
        content_image = content_images[i]
        style_image = style_images[i % len(style_images)]  # Cycle through styles if fewer styles

        # For demo purposes, applying a style transfer function
        stylized_image = content_image * 0.5 + style_image * 0.5  # Simplified blend for demonstration

        # Plot content, style, and stylized images
        axes[i, 0].imshow(content_image)
        axes[i, 0].set_title("Content Image")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(style_image)
        axes[i, 1].set_title("Style Image")
        axes[i, 1].axis('off')

        axes[i, 2].imshow(stylized_image)
        axes[i, 2].set_title("Stylized Output")
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

# Visualize results with different samples
visualize_content_style_output(content_test, style_images)


Loading content images...


 68%|██████▊   | 685/1000 [00:12<00:04, 67.36it/s]